# Primer Parcial - Análisis Predictivo y Recomendación de Estrategia de Finales (1º Final vs 2º Final)

**Objetivo:** Desarrollar un algoritmo basado en Machine Learning y análisis empírico para determinar si a un alumno le conviene presentarse al **Primer Final (1F)** o saltearlo e ir al **Segundo Final (2F)**, basándose en su perfil académico (Firma, parciales, asistencia y talleres).

### Contexto del Reglamento FIUNA:
- `1F-X`: El alumno rindió el 1º Final. Si $X \ge 2$, aprobó. Si $X=1$, reprobó pero mantiene oportunidad para 2F.
- `2F-X`: El alumno rindió el 2º Final. Si $X \ge 2$, aprobó. Si no rindió 1F, fue su única oportunidad.
- `1F-1,2F-X`: El alumno reprobó el 1º Final ($1F-1$) y luego rindió el 2º Final.
- **Trade-off Estratégico:** Presentarse al 1F ofrece 2 intentos acumulados, pero si el alumno está poco preparado corre riesgo de aplazo ($1F-1$). Ir directo a 2F otorga 2-3 semanas más de preparación pero reduce el margen a 1 solo intento.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")

## 1. Mapeo de Carreras y Parsing Vectorizado de Exámenes Finales

Extraemos de forma vectorizada el historial detallado de rendición de finales desde la columna `Nota.Final`.

In [ ]:
career_code_mapping = {
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean = df_raw.copy()
df_clean['Carrera'] = df_clean['Firma'].astype(str).str.strip().map(career_code_mapping)

# Parsing vectorizado de Nota.Final
s_nota = df_clean['Nota.Final'].fillna('')
df_clean['Rendio_1F'] = s_nota.str.contains('1F-').astype(int)
df_clean['Nota_1F'] = s_nota.str.extract(r'1F-(\d)')[0].astype(float)
df_clean['Aprobo_1F'] = (df_clean['Nota_1F'] >= 2).astype(int)

df_clean['Rendio_2F'] = s_nota.str.contains('2F-').astype(int)
df_clean['Nota_2F'] = s_nota.str.extract(r'2F-(\d)')[0].astype(float)
df_clean['Aprobo_2F'] = (df_clean['Nota_2F'] >= 2).astype(int)

df_clean['Nota_Num'] = (df_clean['Nota.Final'].astype(str).str.extractall(r'(\d+)')[0].groupby(level=0).last().astype(float))
df_clean.loc[df_clean['Nota.Final'].isna(), 'Nota_Num'] = np.nan
df_clean['Aprobo_Cualquiera'] = (df_clean['Nota_Num'] >= 2).astype(int)

print("Parsing completado con éxito:")
print(f"- Alumnos que rindieron 1º Final: {df_clean['Rendio_1F'].sum():,}")
print(f"- Alumnos que rindieron 2º Final: {df_clean['Rendio_2F'].sum():,}")

## 2. Análisis Exploratorio de Estrategias (EDA)

Analizamos empíricamente cómo influye el puntaje de **Firma** en la probabilidad de éxito de rendir el 1º Final vs ir directo al 2º Final.

In [ ]:
df_firma = df_clean[df_clean['Firma'] > 0].copy()
df_firma['Firma_Rango'] = pd.cut(df_firma['Firma'], bins=[0, 40, 50, 60, 70, 80, 90, 100])

# Resumen estadistico por tramos de Firma
resumen_firma = df_firma.groupby('Firma_Rango', observed=False).agg(
    Total_Alumnos=('ALUMNO_ID', 'count'),
    Rendieron_1F=('Rendio_1F', 'sum'),
    Tasa_Aprobo_1F=('Aprobo_1F', lambda x: x[df_firma.loc[x.index, 'Rendio_1F'] == 1].mean() * 100),
    Directo_2F=('Rendio_2F', lambda x: ((df_firma.loc[x.index, 'Rendio_1F'] == 0) & (df_firma.loc[x.index, 'Rendio_2F'] == 1)).sum()),
    Tasa_Directo_2F=('Aprobo_2F', lambda x: x[(df_firma.loc[x.index, 'Rendio_1F'] == 0) & (df_firma.loc[x.index, 'Rendio_2F'] == 1)].mean() * 100)
).round(2)

print("=== TASA DE APROBACION POR TRAMOS DE FIRMA ===")
print(resumen_firma)

# Grafico comparativo de Tasa de Aprobacion
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

rango_labels = [str(c) for c in resumen_firma.index]
x = np.arange(len(rango_labels))
width = 0.35

ax[0].bar(x - width/2, resumen_firma['Tasa_Aprobo_1F'], width, label='Aprobación en 1F (%)', color='#2b5c8f')
ax[0].bar(x + width/2, resumen_firma['Tasa_Directo_2F'], width, label='Aprobación Directo 2F (%)', color='#e06d53')
ax[0].set_xlabel('Rango de Puntaje de Firma', fontsize=12)
ax[0].set_ylabel('Tasa de Aprobación (%)', fontsize=12)
ax[0].set_title('Tasa de Aprobación: Rendir 1F vs Ir Directo a 2F', fontsize=14, fontweight='bold')
ax[0].set_xticks(x)
ax[0].set_xticklabels(rango_labels, rotation=30)
ax[0].legend()
ax[0].set_ylim(0, 105)

# Distribución de participantes en 1F vs Directo 2F
ax[1].plot(x, resumen_firma['Rendieron_1F'], marker='o', linewidth=2.5, label='Rendieron 1F', color='#2b5c8f')
ax[1].plot(x, resumen_firma['Directo_2F'], marker='s', linewidth=2.5, label='Fueron Directo a 2F', color='#e06d53')
ax[1].set_xlabel('Rango de Puntaje de Firma', fontsize=12)
ax[1].set_ylabel('Cantidad de Alumnos', fontsize=12)
ax[1].set_title('Cantidad de Alumnos por Decisión y Firma', fontsize=14, fontweight='bold')
ax[1].set_xticks(x)
ax[1].set_xticklabels(rango_labels, rotation=30)
ax[1].legend()

plt.tight_layout()
plt.show()

## 3. Entrenamiento y Evaluación de Modelos Predictivos

Entrenamos clasificadores supervisados (`RandomForestClassifier`) para estimar:
1. **`Modelo_1F`**: Probabilidad de aprobar si se presenta al 1º Final ($P(1F \ge 2)$).
2. **`Modelo_2F_Directo`**: Probabilidad de aprobar si saltea 1F y va directo al 2º Final ($P(2F \ge 2)$).

In [ ]:
features_num = ['Firma', 'Primer.Par', 'Segundo.Par', 'Tercer.Par', 'TPLab.', 'Lab.', 'Proy.', 'Asis', 'Pond.PP', 'Pond.SP']

# 1. Modelo 1F: Alumnos que rindieron 1F
df_1f = df_firma[df_firma['Rendio_1F'] == 1].copy()
X_1f = df_1f[features_num].fillna(0)
y_1f = df_1f['Aprobo_1F']

X_train_1f, X_test_1f, y_train_1f, y_test_1f = train_test_split(X_1f, y_1f, test_size=0.2, random_state=42)
model_1f = RandomForestClassifier(n_estimators=120, random_state=42, max_depth=12)
model_1f.fit(X_train_1f, y_train_1f)

y_pred_1f = model_1f.predict(X_test_1f)
y_prob_1f = model_1f.predict_proba(X_test_1f)[:, 1]
auc_1f = roc_auc_score(y_test_1f, y_prob_1f)

# 2. Modelo 2F Directo: Alumnos que fueron directo a 2F
df_2f_dir = df_firma[(df_firma['Rendio_1F'] == 0) & (df_firma['Rendio_2F'] == 1)].copy()
X_2f_dir = df_2f_dir[features_num].fillna(0)
y_2f_dir = df_2f_dir['Aprobo_2F']

X_train_2f, X_test_2f, y_train_2f, y_test_2f = train_test_split(X_2f_dir, y_2f_dir, test_size=0.2, random_state=42)
model_2f = RandomForestClassifier(n_estimators=120, random_state=42, max_depth=12)
model_2f.fit(X_train_2f, y_train_2f)

y_pred_2f = model_2f.predict(X_test_2f)
y_prob_2f = model_2f.predict_proba(X_test_2f)[:, 1]
auc_2f = roc_auc_score(y_test_2f, y_prob_2f)

print(f"=== RENDIMIENTO DE MODELOS PREDICTIVOS ===")
print(f"Modelo 1º Final - ROC-AUC: {auc_1f:.4f}")
print(f"Modelo 2º Final Directo - ROC-AUC: {auc_2f:.4f}")

# Visualización de Curvas ROC e Importancia de Features
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

fpr1, tpr1, _ = roc_curve(y_test_1f, y_prob_1f)
fpr2, tpr2, _ = roc_curve(y_test_2f, y_prob_2f)

ax[0].plot(fpr1, tpr1, label=f'Modelo 1F (AUC = {auc_1f:.3f})', color='#2b5c8f', linewidth=2)
ax[0].plot(fpr2, tpr2, label=f'Modelo 2F Directo (AUC = {auc_2f:.3f})', color='#e06d53', linewidth=2)
ax[0].plot([0, 1], [0, 1], 'k--', label='Azar')
ax[0].set_title('Curvas ROC - Modelos de Aprobación de Finales', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
ax[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax[0].legend()

importances = pd.Series(model_1f.feature_importances_, index=features_num).sort_values(ascending=True)
importances.plot(kind='barh', ax=ax[1], color='#2b5c8f')
ax[1].set_title('Importancia de Variables (Modelo 1F)', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Importancia Relativa')

plt.tight_layout()
plt.show()

## 4. Algoritmo y Función de Recomendación Estratégica (`recomendar_estrategia`)

Implementamos la función que recibe los antecedentes académicos de cualquier alumno y emite una recomendación fundada.

In [ ]:
def recomendar_estrategia(alumno_dict):
    """
    Evalúa los datos académicos de un alumno y determina la mejor recomendación para rendir el final.
    """
    df_inst = pd.DataFrame([alumno_dict])[features_num].fillna(0)
    p_1f = model_1f.predict_proba(df_inst)[0, 1] * 100
    p_2f = model_2f.predict_proba(df_inst)[0, 1] * 100
    
    firma_val = alumno_dict.get('Firma', 0)
    
    if p_1f >= 50.0:
        rec = "RENDIR 1º FINAL (1F)"
        just = f"Tu probabilidad de aprobar en 1F es alta ({p_1f:.1f}%). Rendir 1F aprovecha tus 2 intentos posibles."
    elif p_2f > p_1f + 8.0:
        rec = "ESPERAR AL 2º FINAL (2F)"
        just = f"Tu probabilidad en 1F es moderada/baja ({p_1f:.1f}%), pero se incrementa a {p_2f:.1f}% para 2F con semanas extra de estudio."
    else:
        rec = "REFORZAR ESTUDIO INTENSIVO"
        just = f"Probabilidades reducidas en ambos exámenes (1F: {p_1f:.1f}%, 2F: {p_2f:.1f}%). Se recomienda repaso profundo previo."
        
    return {
        'Recomendación': rec,
        'Probabilidad 1F (%)': round(p_1f, 1),
        'Probabilidad 2F Directo (%)': round(p_2f, 1),
        'Justificación': just
    }

print("=== DEMOSTRACION DEL ALGORITMO CON CASOS DE PRUEBA ===")

casos_demostracion = [
    {"Nombre": "Alumno Sobresaliente (Firma 85)", "Firma": 85, "Primer.Par": 80, "Segundo.Par": 85, "Tercer.Par": 0, "TPLab.": 90, "Lab.": 90, "Proy.": 0, "Asis": 1, "Pond.PP": 30, "Pond.SP": 35},
    {"Nombre": "Alumno Medio (Firma 62)", "Firma": 62, "Primer.Par": 55, "Segundo.Par": 65, "Tercer.Par": 0, "TPLab.": 80, "Lab.": 80, "Proy.": 0, "Asis": 1, "Pond.PP": 20, "Pond.SP": 25},
    {"Nombre": "Alumno Ajustado / Firma Baja (Firma 42)", "Firma": 42, "Primer.Par": 35, "Segundo.Par": 45, "Tercer.Par": 0, "TPLab.": 70, "Lab.": 70, "Proy.": 0, "Asis": 1, "Pond.PP": 15, "Pond.SP": 20}
]

for c in casos_demostracion:
    print(f"\n--- {c['Nombre']} ---")
    res = recomendar_estrategia(c)
    for k, v in res.items():
        print(f"  {k}: {v}")